In [5]:
import pandas as pd
import requests
import json


Initial start to data collection pulling the entire dataset 

In [2]:
#Website were puklling the endpouint from to get the data 
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

response = requests.get(url, params={"$limit": 5}, timeout=60)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

display(df)

,unique_key,created_date,agency,agency_name,complaint_type,descriptor,location_type,status,community_board,police_precinct,...,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,council_district,bbl,descriptor_2
0,70765514,2026-10-16T01:51:34.000,NYPD,New York City Police Department,Noise - Commercial,Loud Music/Party,Store/Commercial,Unspecified,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,70765005,2026-10-16T01:51:28.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,In Progress,01 QUEENS,Precinct 114,...,34 AVENUE,35 AVENUE,34 AVENUE,35 AVENUE,ADDRESS,ASTORIA,12 STREET,26,4005230002,NaN
2,70770337,2026-10-16T01:51:27.000,NYPD,New York City Police Department,Noise - Residential,Loud Talking,Residential Building/House,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,70769727,2026-10-16T01:51:23.000,DSNY,Department of Sanitation,Residential Disposal Complaint,Storage Area Not Provided,Sidewalk,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Super or Building Manager
4,70766819,2026-10-16T01:51:15.000,DSNY,Department of Sanitation,Dirty Condition,Trash,Sidewalk,In Progress,0 Unspecified,Unspecified,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Littering


In [ ]:
#Filter for street conditions only using parameters DOT( department of transportation) and 
#only street condition issues 
#Only pulling the first 5 in this request.get
params = {
    "$where": "agency = 'DOT' AND complaint_type = 'Street Condition'",
    "$limit": 5
}

response = requests.get(url, params=params, timeout=60)



df = pd.DataFrame(response.json())

display(df[["unique_key", "agency", "complaint_type", "descriptor"]])



,unique_key,agency,complaint_type,descriptor
0,69177427,DOT,Street Condition,Pothole
1,69185057,DOT,Street Condition,Pothole
2,68168575,DOT,Street Condition,Pothole
3,68179684,DOT,Street Condition,Pothole
4,68176123,DOT,Street Condition,Pothole


January Dataset first to depict what the dataset will look like for future months using the api

In [4]:
#Collecting time stamps from January 20205 

params = {
    "$where": (
        "agency = 'DOT' "
        "AND complaint_type = 'Street Condition' "
        "AND created_date >= '2025-01-01T00:00:00' "
        "AND created_date < '2025-02-01T00:00:00'"
    ),
    "$limit": 5
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

df = pd.DataFrame(response.json())

display(df[["created_date", "complaint_type", "descriptor"]])

,created_date,complaint_type,descriptor
0,2025-01-01T02:27:33.000,Street Condition,Pothole
1,2025-01-01T02:31:08.000,Street Condition,Pothole
2,2025-01-01T02:39:44.000,Street Condition,Dumpster - Construction Waste
3,2025-01-01T02:55:52.000,Street Condition,Pothole
4,2025-01-01T04:20:00.000,Street Condition,Wear & Tear


In [ ]:
#matching requests in January total amount of complaints in January 
count_params = {
    "$where": params["$where"],
    "$select": "count(*) AS total_requests"
}

response = requests.get(url, params=count_params, timeout=60)
response.raise_for_status()

display(pd.DataFrame(response.json()))

,total_requests
0,5474


In [ ]:
#Printing the row and column amount 
collect_params = {
    "$where": params["$where"],
    "$order": "created_date ASC, unique_key ASC",
    "$limit": 6000
}

response = requests.get(url, params=collect_params, timeout=120)
response.raise_for_status()

january_df = pd.DataFrame(response.json())

print("Rows:", january_df.shape[0])
print("Columns:", january_df.shape[1])

Rows: 5474
Columns: 36


In [7]:
#Checking for duplicates 

print("Duplicate request IDs:",
      january_df["unique_key"].duplicated().sum())

print("\nColumn names:")
print(january_df.columns.tolist())

Duplicate request IDs: 0

Column names:
['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'address_type', 'city', 'facility_type', 'status', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'police_precinct', 'borough', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'intersection_street_1', 'intersection_street_2', 'council_district', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'latitude', 'longitude', 'location', 'descriptor_2', 'location_type', 'landmark', 'bbl']


In [9]:
#Calculate missing values 
display(january_df.isna().sum())

missing_closure = january_df[
    january_df["closed_date"].isna()
]

display(missing_closure["status"].value_counts())

unique_key                           0
created_date                         0
closed_date                         85
agency                               0
agency_name                          0
complaint_type                       0
descriptor                           0
incident_zip                        68
incident_address                  1550
street_name                       1550
cross_street_1                    1572
cross_street_2                    1571
address_type                         0
city                               566
facility_type                     1723
status                               0
resolution_description               3
resolution_action_updated_date       3
community_board                      0
police_precinct                      0
borough                              0
open_data_channel_type               0
park_facility_name                   0
park_borough                         0
intersection_street_1             2184
intersection_street_2    

status
Pending    85
Name: count, dtype: int64

In [11]:
#Convert text into actual dates we can use 

january_df["created_date"] = pd.to_datetime(
    january_df["created_date"]
)

january_df["closed_date"] = pd.to_datetime(
    january_df["closed_date"]
)

display(january_df[["created_date", "closed_date"]].dtypes)

created_date    datetime64[us]
closed_date     datetime64[us]
dtype: object

In [14]:
#Calculating time difference between openand closure, alongside negative closure times
time_difference = (
    january_df["closed_date"] - january_df["created_date"]
)

january_df["closure_hours"] = (
    time_difference.dt.total_seconds() / 3600
)

display(
    january_df[["created_date", "closed_date", "closure_hours"]].head(10)
)

#check for negative times 
negative_times = january_df["closure_hours"] < 0

print("Negative closure times:", negative_times.sum())

display(january_df["closure_hours"].describe())


,created_date,closed_date,closure_hours
0,2025-01-01 02:27:33,2025-01-02 12:40:00,34.207500
1,2025-01-01 02:31:08,2025-01-06 10:20:00,127.814444
2,2025-01-01 02:39:44,2025-01-02 11:30:02,32.838333
3,2025-01-01 02:55:52,2025-01-06 11:00:00,128.068889
4,2025-01-01 04:20:00,NaT,NaN
5,2025-01-01 04:28:31,2025-01-02 12:03:27,31.582222
6,2025-01-01 06:00:17,2025-01-02 20:10:00,38.161944
7,2025-01-01 06:01:21,2025-01-02 20:45:00,38.727500
8,2025-01-01 06:02:48,2025-01-02 21:35:00,39.536667
9,2025-01-01 06:03:33,2025-01-02 22:05:00,40.024167


Negative closure times: 0


count     5389.000000
mean        50.481935
std        321.988083
min          0.000000
25%         18.547222
50%         23.613333
75%         42.217222
max      12122.434444
Name: closure_hours, dtype: float64

In [ ]:
#Explanation for the mean being 50 hours since the largest time differrences span into 2026 
#May reflect administration oversight or some error ?
display(
    january_df.sort_values("closure_hours", ascending=False)[
        ["created_date", "closed_date", "descriptor", "status", "closure_hours"]
    ].head(5)
)

,created_date,closed_date,descriptor,status,closure_hours
2552,2025-01-14 08:43:53,2026-06-03 11:09:57,Cave-in,Closed,12122.434444
674,2025-01-04 23:12:14,2026-01-07 11:00:00,Pothole,Closed,8819.796111
155,2025-01-01 15:13:35,2026-01-02 11:25:00,Pothole,Closed,8780.190278
2583,2025-01-14 13:01:02,2026-01-14 16:25:00,Pothole,Closed,8763.399444
1314,2025-01-07 16:09:48,2025-12-02 09:18:15,Blocked - Construction,Closed,7889.140833


In [16]:
#Our target variable (binary classification over the 48 hour mark or not)

over_48_hours = january_df["closure_hours"] > 48

print("Recorded closures after 48 hours:", over_48_hours.sum())

Recorded closures after 48 hours: 1138


In [19]:
#copy of january data
january_backup = january_df.copy()
data_folder = Path("../data")
data_folder.mkdir(exist_ok=True)

january_backup.to_csv(
    data_folder / "street_conditions_january_2025.csv",
    index=False
)

print("Data saved!")

Data saved!


In [22]:

with open("../data/street_conditions_january_2025_raw.json", "w") as file:
    json.dump(response.json(), file, indent=2)

print(len(response.json()))
print("Raw JSON saved!")

5474
Raw JSON saved!


February Dataset follows the same format above 

In [23]:
#February collection

start_date = "2025-02-01T00:00:00"
end_date = "2025-03-01T00:00:00"


february_filter = (
    "agency = 'DOT' "
    "AND complaint_type = 'Street Condition' "
    f"AND created_date >= '{start_date}' "
    f"AND created_date < '{end_date}'"
)

response = requests.get(
    url,
    params={
        "$where": february_filter,
        "$select": "count(*) AS total_requests"
    },
    timeout=120
)
response.raise_for_status()

display(pd.DataFrame(response.json()))


,total_requests
0,5623


In [24]:
#Collecting February's requests 

february_records = []
offset = 0
batch_size = 5000

while True:
    response = requests.get(
        url,
        params={
            "$where": february_filter,
            "$order": "created_date ASC, unique_key ASC",
            "$limit": batch_size,
            "$offset": offset
        },
        timeout=120
    )
    response.raise_for_status()

    batch = response.json()
    february_records.extend(batch)

    print("Records collected:", len(february_records))

    if len(batch) < batch_size:
        break

    offset += batch_size

february_df = pd.DataFrame(february_records)

Records collected: 5000
Records collected: 5623


In [25]:
#save February records

with open("../data/street_conditions_february_2025_raw.json", "w") as file:
    json.dump(february_records, file, indent=2)

print("February raw data saved!")

February raw data saved!


In [27]:
#Duplicates 
print("Duplicate IDs:",
      february_df["unique_key"].duplicated().sum())

february_df["created_date"] = pd.to_datetime(february_df["created_date"])
february_df["closed_date"] = pd.to_datetime(february_df["closed_date"])

time_difference = (
    february_df["closed_date"] - february_df["created_date"]
)

february_df["closure_hours"] = (
    time_difference.dt.total_seconds() / 3600
)

print("Missing closure dates:",
      february_df["closed_date"].isna().sum())

print("Negative closure times:",
      (february_df["closure_hours"] < 0).sum())


display(
    february_df.loc[
        february_df["closure_hours"] < 0,
        ["unique_key", "created_date", "closed_date",
         "descriptor", "status", "closure_hours"]
    ]
)

Duplicate IDs: 0
Missing closure dates: 140
Negative closure times: 1


,unique_key,created_date,closed_date,descriptor,status,closure_hours
658,63980274,2025-02-04 14:05:33,2025-02-04 12:25:00,Pothole,Closed,-1.675833


In [30]:
#Cleaned rows February , one row exlcuded from having negative closure time 

february_clean = february_df[
    february_df["closure_hours"].isna()
    | (february_df["closure_hours"] >= 0)
].copy()

print("Rows retained:", len(february_clean))


february_clean.to_csv(    "../data/street_conditions_february_2025_clean.csv",    index=False)
print("February CSV saved!")

Rows retained: 5622
February CSV saved!


Decided to  create a function to mass collect the datasets instead of indivdually pulling them since 
it would be tedious to continue the same process 10 more times

In [31]:
#Automate the process for March-Dec 

def collect_month(start_date, end_date):
    month_filter = (
        "agency = 'DOT' "
        "AND complaint_type = 'Street Condition' "
        f"AND created_date >= '{start_date}' "
        f"AND created_date < '{end_date}'"
    )

    records = []
    offset = 0

    while True:
        response = requests.get(
            url,
            params={
                "$where": month_filter,
                "$order": "created_date ASC, unique_key ASC",
                "$limit": 5000,
                "$offset": offset
            },
            timeout=120
        )
        response.raise_for_status()

        batch = response.json()
        records.extend(batch)

        print("Records collected:", len(records))

        if len(batch) < 5000:
            break

        offset += 5000

    return records

Used march as the start in oder to verify the function worked properly

In [32]:
march_records = collect_month(
    "2025-03-01T00:00:00",
    "2025-04-01T00:00:00"
)

march_df = pd.DataFrame(march_records)

Records collected: 5000
Records collected: 7035


In [33]:
#March Records 

with open("../data/street_conditions_march_2025_raw.json", "w") as file:
    json.dump(march_records, file, indent=2)

# Check duplicate IDs
print("Duplicate IDs:",
      march_df["unique_key"].duplicated().sum())

# Convert dates and calculate closure hours
march_df["created_date"] = pd.to_datetime(march_df["created_date"])
march_df["closed_date"] = pd.to_datetime(march_df["closed_date"])

time_difference = march_df["closed_date"] - march_df["created_date"]
march_df["closure_hours"] = time_difference.dt.total_seconds() / 3600

print("Missing closure dates:",
      march_df["closed_date"].isna().sum())

print("Negative closure times:",
      (march_df["closure_hours"] < 0).sum())

Duplicate IDs: 0
Missing closure dates: 212
Negative closure times: 2


In [34]:
display(
    march_df.loc[
        march_df["closure_hours"] < 0,
        ["unique_key", "created_date", "closed_date",
         "descriptor", "status", "closure_hours"]
    ]
)

,unique_key,created_date,closed_date,descriptor,status,closure_hours
2682,64345550,2025-03-12 06:05:50,2025-03-12 05:45:00,Pothole,Closed,-0.347222
6203,64490612,2025-03-27 15:53:45,2025-03-27 13:00:00,Pothole,Closed,-2.895833


In [35]:
#clean march records

march_clean = march_df[
    march_df["closure_hours"].isna()
    | (march_df["closure_hours"] >= 0)
].copy()

march_clean.to_csv(
    "../data/street_conditions_march_2025_clean.csv",
    index=False
)

print("March rows retained:", len(march_clean))

March rows retained: 7033


Decided to create another function for the cleaning process to make everuthing faster usingthe collect month function created as a helper consolidating everything 

In [36]:
#Easier comapct version to clean the rest ofthe data 

def process_month(year, month):
    # Set the date range, including December's year change
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.DateOffset(months=1)

    records = collect_month(
        start.isoformat(),
        end.isoformat()
    )

    filename = f"street_conditions_{year}_{month:02d}"

    # Preserve original records before cleaning
    with open(f"../data/{filename}_raw.json", "w") as file:
        json.dump(records, file, indent=2)

    df = pd.DataFrame(records)

    duplicates = df["unique_key"].duplicated().sum()

    # Stop for investigation rather than silently removing duplicates
    if duplicates > 0:
        raise ValueError(f"{filename}: found {duplicates} duplicate IDs")

    df["created_date"] = pd.to_datetime(df["created_date"])
    df["closed_date"] = pd.to_datetime(df["closed_date"])

    difference = df["closed_date"] - df["created_date"]
    df["closure_hours"] = difference.dt.total_seconds() / 3600

    missing = df["closed_date"].isna().sum()
    negative = (df["closure_hours"] < 0).sum()

    # Keep a record of excluded rows
    df[df["closure_hours"] < 0].to_csv(
        f"../data/{filename}_excluded.csv", index=False
    )

    clean = df[
        df["closure_hours"].isna()
        | (df["closure_hours"] >= 0)
    ].copy()

    clean.to_csv(f"../data/{filename}_clean.csv", index=False)

    return {
        "month": month,
        "collected": len(df),
        "duplicates": duplicates,
        "missing_closure": missing,
        "negative_excluded": negative,
        "retained": len(clean)
    }

In [37]:
monthly_checks = []

for month in range(4, 13):
    print(f"\nCollecting month {month}")
    summary = process_month(2025, month)
    monthly_checks.append(summary)

checks_df = pd.DataFrame(monthly_checks)

checks_df.to_csv("../data/monthly_checks_april_december_2025.csv",
                 index=False)

display(checks_df)


Records collected: 5000
Records collected: 6659

Records collected: 5000
Records collected: 7449

Records collected: 5000
Records collected: 7243

Records collected: 5000
Records collected: 6138

Records collected: 5000
Records collected: 5590

Records collected: 5000
Records collected: 5411

Records collected: 4829

Records collected: 4049

Records collected: 4830


,month,collected,duplicates,missing_closure,negative_excluded,retained
0,4,6659,0,267,0,6659
1,5,7449,0,596,0,7449
2,6,7243,0,297,0,7243
3,7,6138,0,350,1,6137
4,8,5590,0,376,0,5590
5,9,5411,0,360,0,5411
6,10,4829,0,359,0,4829
7,11,4049,0,199,0,4049
8,12,4830,0,236,2,4828


In [3]:
from pathlib import Path

for file in sorted(Path("../data").glob("*.csv")):
    print(file.name)

monthly_checks_april_december_2025.csv
street_conditions_april_2025_clean.csv
street_conditions_august_2025_clean.csv
street_conditions_december_2025_clean_clean.csv
street_conditions_february_2025_clean.csv
street_conditions_january_2025.csv
street_conditions_july_2025_clean.csv
street_conditions_june_2025_clean.csv
street_conditions_march_2025_clean.csv
street_conditions_may_2025_clean.csv
street_conditions_november_2025_clean.csv
street_conditions_october_2025_clean.csv
street_conditions_september_2025_clean.csv


In [ ]:
#Combining all of the months datasets to be in one csv file 
monthly_files = sorted(
    Path("../data").glob("street_conditions_*2025*.csv")
)

monthly_frames = []

for file in monthly_files:
    df = pd.read_csv(file, dtype={"unique_key": str, "incident_zip": str})
    monthly_frames.append(df)

all_data = pd.concat(monthly_frames, ignore_index=True)

print("Monthly files:", len(monthly_files))
print("Total rows:", len(all_data))
print("Duplicate IDs:", all_data["unique_key"].duplicated().sum())

all_data.to_csv(
    "../data/street_conditions_2025_combined.csv",
    index=False
)

print("Combined dataset saved!")

Monthly files: 12
Total rows: 70324
Duplicate IDs: 0
Combined dataset saved!


In [9]:
missing_closure = all_data[
    all_data["closed_date"].isna()
]

print("Missing closure dates:", len(missing_closure))

display(missing_closure["status"].value_counts(dropna=False))

Missing closure dates: 3477


status
Pending        3467
Open              8
Unspecified       2
Name: count, dtype: int64

In [10]:
#Unspecified check 

display(
    missing_closure.loc[
        missing_closure["status"] == "Unspecified",
        ["unique_key", "created_date", "descriptor",
         "status", "resolution_description"]
    ]
)

,unique_key,created_date,descriptor,status,resolution_description
35453,65170382,2025-06-05 06:52:08,Pothole,Unspecified,NaN
43801,64307255,2025-03-10 14:47:55,Pothole,Unspecified,The Department of Transportation assigned this...


In [ ]:
model_data = all_data.copy()

model_data["closed_48h"] = pd.Series(
    pd.NA,
    index=model_data.index,
    dtype="Int64"
)


In [12]:
#Binary assignment if its been closed within 48hrs or after
hours = model_data["closure_hours"]

# Known closure within 48 hours
model_data.loc[
    hours.notna() & (hours >= 0) & (hours <= 48),
    "closed_48h"
] = 1

# Known closure after 48 hours
model_data.loc[hours > 48, "closed_48h"] = 0

# Missing closure date and status recorded as Pending or Open
model_data.loc[
    model_data["closed_date"].isna()
    & model_data["status"].isin(["Pending", "Open"]),
    "closed_48h"
] = 0

display(model_data["closed_48h"].value_counts(dropna=False))

closed_48h
1       44355
0       25967
<NA>        2
Name: count, dtype: Int64

In [13]:
#Excluding the two <NA> outcomes 

labeled_data = model_data.dropna(
    subset=["closed_48h"]
).copy()

print("Labeled rows:", len(labeled_data))

Labeled rows: 70322


In [15]:
print("Rows:", labeled_data.shape[0])
print("Columns:", labeled_data.shape[1])

Rows: 70322
Columns: 38


In [14]:
#Final labeled data used to perform training on 
labeled_data.to_csv(
    "../data/street_conditions_2025_labeled.csv",
    index=False
)

In [16]:
hours = labeled_data["closure_hours"].dropna()

for cutoff in [24, 48, 72]:
    print(cutoff, "hours:", (hours <= cutoff).sum(),
          "recorded closures within cutoff")

24 hours: 31101 recorded closures within cutoff
48 hours: 44355 recorded closures within cutoff
72 hours: 51320 recorded closures within cutoff
